In [70]:
import pandas as pd
import numpy as np

In [71]:
# =====================================================
# Load Dataset & Basic Preprocessing
# =====================================================

df = pd.read_csv('../Data/feature-columns.csv')
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.set_index('datetime')

In [72]:
# 80% train, 20% test
split = int(len(df) * 0.8)

train = df.iloc[:split]
test  = df.iloc[split:]

print(f"Train: {train.index[0].date()} to {train.index[-1].date()}")
print(f"Test:  {test.index[0].date()} to {test.index[-1].date()}")
print(f"Train rows: {len(train)}, Test rows: {len(test)}")

Train: 2021-01-08 to 2023-05-28
Test:  2023-05-28 to 2023-12-31
Train rows: 20889, Test rows: 5223


In [73]:
X_train = train.drop(columns=['demand_MW'])
y_train = train['demand_MW']

X_test = test.drop(columns=['demand_MW'])
y_test = test['demand_MW']

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")

X_train shape: (20889, 10)
X_test shape:  (5223, 10)


In [74]:
from xgboost import XGBRegressor 

model = XGBRegressor( 
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=6, 
    subsample=0.8, 
    colsample_bytree=0.8, 
    random_state=42, 
    n_jobs=-1
) 

model.fit( 
    X_train, y_train, 
    eval_set=[(X_test, y_test)],
    verbose=20
)

y_pred = model.predict(X_test)

[0]	validation_0-rmse:1120.74291
[20]	validation_0-rmse:489.54925
[40]	validation_0-rmse:287.02777
[60]	validation_0-rmse:228.01645
[80]	validation_0-rmse:209.18247
[99]	validation_0-rmse:200.34778


In [75]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae  = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
accuracy = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.0f}")   
print(f"RMSE : {rmse:.0f}")   
print("R2  :", accuracy)

MAE  : 156
RMSE : 200
R2  : 0.9649171747830619


In [76]:
import joblib

# Model saved
joblib.dump(model, 'xgboost_electricity_model.pkl')
print("✅ Model saved successfully!")

✅ Model saved successfully!


In [78]:
feature_columns = list(X_train.columns)

joblib.dump(feature_columns, 'feature_columns.pkl')
print("Feature columns saved!")
print(feature_columns)  
print(f"\nTotal features: {len(feature_columns)}")

Feature columns saved!
['lag_1', 'lag_24', 'lag_168', 'rolling_24_mean', 'rolling_168_mean', 'hour', 'day_of_week', 'weekend', 'holiday', 'temperature']

Total features: 10
